# Day 3 — Tune, multi-lead, skill curve

Reuse Day 1 data → train a small CNN at leads **1, 3, 6, 9** → plot model vs persistence vs climatology.

**Goal:** skill curve image saved; model beats persistence at least at the shorter leads.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarib-sami/ninonet/blob/main/enso/day3_skill.ipynb)

Runtime → Change runtime type → **GPU** (optional but nicer).

If behind: run leads **3 and 6 only** (edit `LEADS` below).


## 0. Install


In [ ]:
# torch comes preinstalled on Colab
!pip install -q xarray netCDF4 numpy pandas scikit-learn matplotlib


## 1. Mount Drive and load Day 1 files


In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/ensocast/data")
OUT_DIR = Path("/content/drive/MyDrive/ensocast/artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data:", DATA_DIR)
print("Artifacts:", OUT_DIR)
assert (DATA_DIR / "pacific_anom.nc").exists(), "Missing pacific_anom.nc — rerun Day 1"
assert (DATA_DIR / "oni_monthly.csv").exists(), "Missing oni_monthly.csv — rerun Day 1" 


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

anom = xr.open_dataarray(DATA_DIR / "pacific_anom.nc")
if isinstance(anom, xr.Dataset):
    anom = anom[list(anom.data_vars)[0]]

oni_df = pd.read_csv(DATA_DIR / "oni_monthly.csv", parse_dates=["time"])

arr = anom.values.astype("float32")  # (time, lat, lon)
times = pd.to_datetime(anom["time"].values).to_period("M").to_timestamp()

oni_series = oni_df.set_index("time")["oni"]
oni_series.index = pd.to_datetime(oni_series.index).to_period("M").to_timestamp()
oni = oni_series.reindex(times).to_numpy(dtype="float32")

missing = int(np.isnan(oni).sum())
if missing:
    print(f"Dropping {missing} SST month(s) with no ONI.")
    valid = ~np.isnan(oni)
    arr, times, oni = arr[valid], times[valid], oni[valid]

assert not np.isnan(oni).any()
print("aligned months:", len(arr), "grid:", arr.shape[1:])
print("range:", times[0].date(), "→", times[-1].date())


## 2. Helpers: samples, split, metrics

Same rules as Day 2, but packaged so we can loop over leads.


In [ ]:
from sklearn.metrics import mean_squared_error

WINDOW = 12
# If behind on time, use [3, 6] only
LEADS = [1, 3, 6, 9]

TRAIN_END = "2005-12-31"
VAL_START, VAL_END = "2006-01-01", "2015-12-31"
TEST_START = "2016-01-01"


def build_samples(arr, times, oni, window=WINDOW, lead=3):
    # X = 12 maps ending at t; y = ONI at t+lead; oni_at_t for persistence
    X_list, y_list, t_end_list, oni_at_t_list = [], [], [], []
    for t in range(window - 1, len(arr) - lead):
        X_list.append(arr[t - window + 1 : t + 1])
        y_list.append(oni[t + lead])
        t_end_list.append(times[t])
        oni_at_t_list.append(oni[t])
    X = np.stack(X_list).astype("float32")
    y = np.array(y_list, dtype="float32")
    t_end = pd.to_datetime(t_end_list)
    oni_at_t = np.array(oni_at_t_list, dtype="float32")
    return X, y, t_end, oni_at_t


def time_masks(t_end):
    train = t_end <= TRAIN_END
    val = (t_end >= VAL_START) & (t_end <= VAL_END)
    test = t_end >= TEST_START
    return train, val, test


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def corr(y_true, y_pred):
    if np.std(y_true) < 1e-8 or np.std(y_pred) < 1e-8:
        return float("nan")
    return float(np.corrcoef(y_true, y_pred)[0, 1])


print("LEADS:", LEADS)


## 3. Small CNN (Day 3 tweaks)

Still tiny. Changes vs Day 2: slightly more dropout + Adam weight decay to reduce memorizing.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


class ENSOForecaster(nn.Module):
    def __init__(self, in_months=12, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_months, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def make_loader(X, y, mask, batch_size=32, shuffle=False):
    ds = TensorDataset(torch.from_numpy(X[mask]), torch.from_numpy(y[mask]))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


print("params (example):", sum(p.numel() for p in ENSOForecaster().parameters()))


## 4. Train + evaluate one lead

Early stop on validation RMSE; restore best weights; score test vs persistence / climatology.


In [ ]:
def eval_model(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            pred = model(xb.to(device)).cpu().numpy()
            ys.append(yb.numpy())
            ps.append(pred)
    yt, yp = np.concatenate(ys), np.concatenate(ps)
    return rmse(yt, yp), corr(yt, yp), yt, yp


def train_one_lead(
    X, y, train_mask, val_mask,
    epochs=40, patience=8, lr=1e-3, weight_decay=1e-4, seed=0,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = ENSOForecaster().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    train_loader = make_loader(X, y, train_mask, shuffle=True)
    val_loader = make_loader(X, y, val_mask, shuffle=False)

    best_val = float("inf")
    best_state = None
    stall = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
            n += len(xb)
        train_mse = total / n

        val_rmse, val_corr, _, _ = eval_model(model, val_loader)
        print(
            f"  epoch {epoch:02d}  train_mse={train_mse:.4f}  "
            f"val_rmse={val_rmse:.3f}  val_corr={val_corr:.3f}"
        )

        if val_rmse < best_val - 1e-4:
            best_val = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stall = 0
        else:
            stall += 1
            if stall >= patience:
                print(f"  early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    return model, best_val


print("train_one_lead ready")


## 5. Loop over leads

For each lead: build samples → baselines → train → test metrics.


In [ ]:
results = []  # one dict per lead

for lead in LEADS:
    print("\n" + "=" * 60)
    print(f"LEAD = {lead}")
    print("=" * 60)

    X, y, t_end, oni_at_t = build_samples(arr, times, oni, window=WINDOW, lead=lead)
    train_mask, val_mask, test_mask = time_masks(t_end)
    print(
        f"samples X{X.shape}  train={train_mask.sum()}  "
        f"val={val_mask.sum()}  test={test_mask.sum()}"
    )

    y_test = y[test_mask]
    pers_test = oni_at_t[test_mask]
    clim_test = np.zeros_like(y_test)

    pers_rmse, pers_corr = rmse(y_test, pers_test), corr(y_test, pers_test)
    clim_rmse, clim_corr = rmse(y_test, clim_test), corr(y_test, clim_test)
    print(f"persistence  RMSE={pers_rmse:.3f}  corr={pers_corr:.3f}")
    print(f"climatology  RMSE={clim_rmse:.3f}  corr={clim_corr:.3f}")

    model, best_val = train_one_lead(X, y, train_mask, val_mask, seed=42 + lead)
    test_loader = make_loader(X, y, test_mask, shuffle=False)
    test_rmse, test_corr, y_true, y_pred = eval_model(model, test_loader)

    beat = test_rmse < pers_rmse
    print(f"\nTEST lead {lead}: model RMSE={test_rmse:.3f} corr={test_corr:.3f}")
    print("✓ beats persistence" if beat else "○ does not beat persistence")

    # save per-lead checkpoint
    ckpt_path = OUT_DIR / f"enso_cnn_lead{lead}.pt"
    torch.save(
        {
            "model_state": model.state_dict(),
            "window": WINDOW,
            "lead": lead,
            "test_rmse": test_rmse,
            "test_corr": test_corr,
            "pers_rmse": pers_rmse,
            "pers_corr": pers_corr,
            "clim_rmse": clim_rmse,
            "clim_corr": clim_corr,
            "best_val_rmse": best_val,
        },
        ckpt_path,
    )
    print("Wrote", ckpt_path)

    results.append(
        {
            "lead": lead,
            "model_rmse": test_rmse,
            "model_corr": test_corr,
            "pers_rmse": pers_rmse,
            "pers_corr": pers_corr,
            "clim_rmse": clim_rmse,
            "clim_corr": clim_corr,
            "beats_persistence": beat,
            "n_test": int(test_mask.sum()),
        }
    )

results_df = pd.DataFrame(results)
print("\n=== SUMMARY ===")
print(results_df.to_string(index=False))


## 6. Skill curve

Honest shape: model beats persistence at short leads, then skill fades. Show both **correlation** (higher better) and **RMSE** (lower better).


In [ ]:
import matplotlib.pyplot as plt

leads = results_df["lead"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(leads, results_df["model_corr"], "o-", label="model", linewidth=2)
ax.plot(leads, results_df["pers_corr"], "s--", label="persistence", alpha=0.85)
ax.plot(leads, results_df["clim_corr"], "^:", label="climatology", alpha=0.85)
ax.set_xlabel("lead (months)")
ax.set_ylabel("correlation")
ax.set_title("Skill curve — correlation (higher better)")
ax.set_xticks(leads)
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(leads, results_df["model_rmse"], "o-", label="model", linewidth=2)
ax.plot(leads, results_df["pers_rmse"], "s--", label="persistence", alpha=0.85)
ax.plot(leads, results_df["clim_rmse"], "^:", label="climatology", alpha=0.85)
ax.set_xlabel("lead (months)")
ax.set_ylabel("RMSE (°C)")
ax.set_title("Skill curve — RMSE (lower better)")
ax.set_xticks(leads)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = OUT_DIR / "skill_curve.png"
fig.savefig(fig_path, dpi=150)
print("Wrote", fig_path)
plt.show()

csv_path = OUT_DIR / "skill_curve.csv"
results_df.to_csv(csv_path, index=False)
print("Wrote", csv_path)

n_beat = int(results_df["beats_persistence"].sum())
print(f"\nBeats persistence on RMSE at {n_beat}/{len(results_df)} leads.")
if n_beat >= 1:
    print("Day 3 checkpoint: skill curve ready.")
else:
    print("No lead beat persistence yet — try LEADS=[3,6], more epochs, or check alignment.")
